# L31 Lab: Classification and Tracking, notebook

This notebook runs the last two stages of the course's radar chain: it classifies a target from its micro-Doppler signature, and it tracks a walker in range and angle. The functions are the provided ones in `radar_lab.py`: `spectrogram`, `features`, `train`, `classify`, `confusion_matrix`, `alpha_beta`, and `monopulse_angle`, with the L30 Lab's `cfar_2d` finding each target first. You do not write them; you run them, change what each task names, and record what you see in Tables D to F of the L31 Lab packet. There are three tasks, (a) to (c), about 30 to 45 minutes in all, and six changed lines between them.

**To run it:** unzip the lab's bundle, `L31_lab.zip` from the L31 page's Notebook Setup frame, into one folder: it holds this notebook, `radar_lab.py`, every `L31_*.npz` this notebook loads, and the data README. Start Jupyter in that folder and run the cells in order from the top.

The recorded data was simulated at the Phaser app's **Range + Velocity** preset: 300 µs chirps of 500 MHz, one every 0.5 ms, at 9.9 GHz, so λ = 3.03 cm.

- `L31_<target>_<n>.npz`, four records each of a fan, a fidget spinner, a waving hand, and a walker: 2.0 s of back-to-back chirps. To keep the files small, each chirp keeps 90 samples (0.3 MSPS): the range bins are still 0.30 m, and they reach 4.4 m. Each file's `note` says what it holds.
- `L31_dismount.npz`, a walker crossing in front of the kit on a diagonal and back, with both subarray channels (`iq_ch`) and the walk's truth. It keeps every 4th chirp (PRI 2 ms), so a 128 ms frame is 64 chirps.

The app's **Save Capture** keeps one burst, 256 chirps or 128 ms, which is too short to hold a stride or a turn. So tasks (a) to (c) run on these records with or without a kit, and the burst you save on mark B goes to the angle cell after Procedure B. A record of your own at least 2 s long, from any source, runs in every cell: change the names in the next cell.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.signal import peak_widths
import radar_lab as rl

# Where the captures are: the course's radar-data/ if this folder sits beside
# it, as in the course repository; otherwise this folder, the unzipped bundle.
DATA = Path("../radar-data") if Path("../radar-data").exists() else Path(".")

KINDS = ["walker", "fan", "spinner", "hand"]
# The records the classifier uses: four of each target. Change the names to
# run on records of your own, each at least 2 s long.
RECORDS = {k: [f"L31_{k}_{n}.npz" for n in range(1, 5)] for k in KINDS}
WALK = "L31_dismount.npz"        # the walk to track: both channels, and its truth

caps = {(k, n): rl.load_capture(DATA / name) for k in KINDS for n, name in enumerate(RECORDS[k], 1)}
walk = rl.load_capture(DATA / WALK)
LAM = rl.C / walk["fc"]
for (k, n), cap in caps.items():
    print(f"{k:8s}{n}  {cap['iq'].shape[0] * cap['PRI']:.1f} s: {str(cap.get('note', ''))[22:]}")
print(f"walk      {walk['iq'].shape[0] * walk['PRI']:.1f} s, PRI {walk['PRI'] * 1e3:.1f} ms, "
      f"{'both channels' if 'iq_ch' in walk else 'no iq_ch: range only'}")

In [ ]:
# Helpers used by the tasks. Run this cell; you do not need to change it.
FRAME = 0.128                    # s: one range-Doppler frame, 256 chirps of Range + Velocity

def clusters(det, hits, rd):
    """One entry per cluster of touching detected cells, the L30 notebook's list:
    (range, velocity, snr_db, n_cells, peak_dbfs), strongest first."""
    lab, _ = ndimage.label(det)
    best = {}
    for (i, j), h in zip(zip(*np.nonzero(det)), hits):
        k = lab[i, j]
        if k not in best or rd[i, j] > best[k][4]:
            best[k] = (*h, int((lab == k).sum()), rd[i, j])
    return sorted(best.values(), key=lambda c: -c[4])

def detect(cap, bias_db=11.6):
    """The L30 chain on a record's first 256 chirps (2-pulse canceller, map, CFAR):
    the strongest detection beyond 0.3 m, as (range, velocity, snr_db, n_cells, peak_dbfs)."""
    burst = {**cap, "iq": cap["iq"][:256]}
    rd, rng, vel = rl.range_doppler(rl.mti(burst, "2pulse"))
    det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), bias_db, axes=(rng, vel))
    return [c for c in clusters(det, hits, rd) if c[0] > 0.3][0]

def target_bins(cap, r, v):
    """The range bins a detection at r m, v m/s (+ closing) covers over the whole
    record: from r out along its velocity, and 0.45 m either side."""
    rax = rl.range_axis(cap)
    r2 = r - v * cap["iq"].shape[0] * cap["PRI"]
    lo, hi = min(r, r2) - 0.45, max(r, r2) + 0.45
    idx = np.flatnonzero((rax >= max(lo, 0.0)) & (rax <= hi))
    return slice(idx[0], idx[-1] + 1)

def max_hold(cap):
    """The app's range-Doppler view over the record, canceller on: each cell's
    largest value over the record's 256-chirp bursts."""
    maps = []
    for k in range(cap["iq"].shape[0] // 256):
        rd, rng, vel = rl.range_doppler(rl.mti({**cap, "iq": cap["iq"][k * 256:(k + 1) * 256]}, "2pulse"))
        maps.append(rd)
    return np.max(maps, axis=0), rng, vel

def line_width(f, S):
    """Half-power width, Hz, of the strongest line in a spectrogram's average spectrum."""
    p = np.mean(10 ** (S / 10), axis=1)
    return peak_widths(p, [np.argmax(p)], rel_height=0.5)[0][0] * (f[1] - f[0])

def leave_one_out(X, y):
    """Classify each record with a classifier trained on all the others."""
    pred = []
    for i in range(len(y)):
        keep = np.arange(len(y)) != i
        pred.append(rl.classify(X[i], rl.train(X[keep], y[keep])))
    return pred

def track_meas(cap, bias_db=11.6, gate=1.0):
    """One measurement per 128 ms frame: the L30 chain on the frame, the detection
    nearest the last one (within gate m) beyond 0.6 m, and its monopulse angle,
    averaged over the frame's chirps. NaN for a frame with none."""
    n = int(round(FRAME / cap["PRI"]))
    rax = rl.range_axis(cap)
    zr, za, prev = [], [], None
    for k in range(cap["iq"].shape[0] // n):
        sl = slice(k * n, (k + 1) * n)
        frame = {**cap, "iq": cap["iq"][sl]}
        if "iq_ch" in cap:
            frame["iq_ch"] = cap["iq_ch"][:, sl]
        frame = rl.mti(frame, "2pulse")
        rd, rng, vel = rl.range_doppler(frame)
        det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), bias_db, axes=(rng, vel))
        near = [c for c in clusters(det, hits, rd)
                if c[0] > 0.6 and (prev is None or abs(c[0] - prev) < gate)]
        if near:
            prev = near[0][0]
            zr.append(prev)
            ib = int(np.argmin(np.abs(rax - prev)))
            za.append(rl.monopulse_angle(frame, ib).mean() if "iq_ch" in frame else np.nan)
        else:
            zr.append(np.nan)
            za.append(np.nan)
    return np.array(zr), np.array(za)

## No kit? Procedures A and B on recorded data

With no kit, do Procedures A and B in the Phaser app's simulator with **Load Recording**, as the L31 page's No Hardware? frame says: it plays these same files through the app's Range-Doppler view. These cells print the numbers the packet's key uses, so check your readings of the view against them. With a kit, they still run, as shipped; skip to the angle cell.

### Procedure A: read the micro-Doppler in the range-Doppler view

The cell draws the range-Doppler view of record 1 of each target, with the 2-pulse canceller on, holding each cell's largest value over the record's eight bursts: what you see if you watch the view for 2 s. It prints the speeds the target reaches, the velocities within 20 dB of the view's peak, and each record's note, which gives the target's size and rate. Predict each target's largest speed from the note (your Pre-Lab Question 1), then record the prediction, the speeds read, and whether the pattern is centered on 0 m/s in Table A.

**Check:** the walker reads from $-1.3$ to $+3.9$ m/s, the fan from $-8.7$ to $+8.7$, the spinner $\pm 5.0$, and the hand $\pm 2.0$. Only the walker is off center. Each edge reads a little past the prediction, because the 20 dB edge includes the Hann window's skirt, about four velocity cells.


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
for ax, k in zip(axs, KINDS):
    cap = caps[(k, 1)]
    rd, rng, vel = max_hold(cap)
    keep = (rng > 0.3) & (rng < 4.4)
    prof = rd[keep].max(axis=0)
    band = vel[prof > prof.max() - 20]
    print(f"{k:8s} {band.min():+5.1f} to {band.max():+5.1f} m/s.  {str(cap['note'])[22:].split(';')[0]}")
    ax.pcolormesh(vel, rng[keep], rd[keep], vmin=prof.max() - 50, vmax=prof.max(), cmap="Blues", shading="auto")
    ax.set_xlim(-12, 12); ax.set_xlabel("velocity (m/s), + closing"); ax.set_title(f"{k}, record 1", fontsize=10)
axs[0].set_ylabel("range (m)")
plt.tight_layout(); plt.show()

### Procedure B: watch the walk

`L31_dismount.npz` is the walk of Part B: from tape mark A, 1.5 m out and 12° to the E1 side of boresight, on a diagonal to B, 4.0 m out and 10° to the E8 side, a turn at B, and back to A, at 1 m/s, past a chair at 3.2 m. The cell runs the L30 chain on three 128 ms frames, near A on the way out, at the turn, and near A on the way back, and prints up to six detections a frame, **sorted by peak power**, strongest first, with each one's SNR. Record the walker's range and velocity in each frame in Table B, and say why the velocity has the sign it has.

**Check:** going out, near 1.7 m, the walker opens at $-0.84$ m/s: the diagonal takes it away from the kit. Coming back, near 2.0 m, it closes at $+0.84$ m/s. At the turn the strongest line is a leg swinging, 3.77 m at $-1.80$ m/s. The **torso** is the 5th line, **4.07 m at $+0.12$ m/s**, one velocity cell from zero, 21.9 dB SNR: record that one in Table B's turn row. Its peak is weaker than the legs' because its radial speed passes through zero at the turn, into the canceller's notch at 0 m/s, though its SNR is among the highest. The chair never appears, because the canceller removes what stands still. These frames hold every 4th chirp, so the view wraps at $\pm 3.8$ m/s instead of the burst's $\pm 15$ m/s; a fast leg can wrap to the other side.


In [ ]:
n = int(round(FRAME / walk["PRI"]))
n_frames = walk["iq"].shape[0] // n
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, (label, k) in zip(axs, (("near A, going out", 1), ("at the turn, B", n_frames // 2), ("near A, coming back", n_frames - 3))):
    sl = slice(k * n, (k + 1) * n)
    rd, rng, vel = rl.range_doppler(rl.mti({**walk, "iq": walk["iq"][sl]}, "2pulse"))
    det, hits = rl.cfar_2d(rd, (2, 2), (4, 4), 11.6, axes=(rng, vel))
    print(f"frame {k} ({(k + 0.5) * FRAME:.2f} s), {label}:")
    for r, v, snr, cells_, peak in clusters(det, hits, rd)[:6]:      # by peak power, strongest first
        print(f"    {r:5.2f} m  {v:+5.2f} m/s  {snr:5.1f} dB SNR  {cells_:2d} cells  peak {peak:6.1f} dBFS")
    keep = rng < 4.4
    ax.pcolormesh(vel, rng[keep], rd[keep], vmin=rd.max() - 50, vmax=rd.max(), cmap="Blues", shading="auto")
    ax.set_xlabel("velocity (m/s), + closing"); ax.set_title(label, fontsize=10)
axs[0].set_ylabel("range (m)")
plt.tight_layout(); plt.show()

### The angle at mark B

Which side of broadside reads a positive angle depends on the kit's channel order, so check it once. The cell takes one record with both subarray channels, `B_REC`, runs the L30 chain on each 128 ms frame of it, and prints the range and the `monopulse_angle` of the farthest detection: for a burst saved on mark B, the angle at B.

- **With a kit:** set `B_REC` to the burst you saved swaying on mark B with **Subarray Channels** on (Procedure B, step 3), and run it. A person standing still is removed by the canceller, so the cell finds nothing in a still record; that is why you sway.
- **With no kit:** run it as shipped, on `L31_dismount.npz`; its farthest frame is the turn at B. It reads the same angle that task (c)'s angle plot shows at the turn.

Record the sign in Table B's last row: a positive angle at B means + is the E8 side.

**Check:** as shipped, the farthest detection is 4.07 m at $+9.9°$, frame 21, 2.75 s into the walk: mark B is $+10°$, to the E8 side, so on this kit + is the E8 side.


In [ ]:
B_REC = WALK        # with a kit: the burst you saved on mark B, e.g. "L31_sway_B.npz"

brec = rl.load_capture(DATA / B_REC)
if "iq_ch" not in brec:
    print(f"{B_REC} has no iq_ch: save it again with Subarray Channels on")
else:
    br, ba = track_meas(brec)
    if np.all(np.isnan(br)):
        print(f"{B_REC}: no moving target found; sway on the mark and save again")
    else:
        kb = int(np.nanargmax(br))
        angle_at_B = float(ba[kb])
        print(f"{B_REC}: farthest detection {br[kb]:.2f} m at {angle_at_B:+.1f} deg, "
              f"frame {kb} ({(kb + 0.5) * FRAME:.2f} s) of {len(br)}")


## Task (a): the spectrogram window

The cell draws the slow-time spectrograms of the spinner and the walker (record 1 of each), over the range bins of each one's L30 detection, with a window of `N_WIN` chirps. It prints the window's length, its resolution $1/T_w$, and three widths, each read at half power from the spectrum averaged over the record: the spinner's strongest line, the walker's torso line, and the walker's spread, the second of the three features.

1. Run it as it stands, `N_WIN = 64`, the lesson's choice for a walker. Record the row in Table D.
2. **Change one line:** set `N_WIN = 16` and run it. Record the row.
3. **Change one line:** set `N_WIN = 256` and run it. Record the row, and compare the three plots.

**Check:** the resolution is 125, 31, and 7.8 Hz. The walker's torso line narrows with the window, from 243 Hz to 57 and then 13 Hz, and its spread falls from 1.29 to 0.68 and 0.51 m/s. At 256 chirps the limb arcs blur along time, as the lesson's Window Length frame predicts for a window 4 times the best one. The spinner's line narrows from 196 Hz to 72, but only to 41 Hz at 256 chirps, five times the resolution: its rate wanders by a few percent while you hold it, and its arms are identical, so its band is a comb of lines 60 Hz apart that a long window starts to separate. The classifier in task (b) uses 64 chirps whatever you leave here.

In [ ]:
N_WIN = 64          # TASK (a): change to 16, then to 256

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
widths = {}
for ax, k in zip(axs, ("spinner", "walker")):
    cap = caps[(k, 1)]
    r, v, *_ = detect(cap)
    t, f, S = rl.spectrogram(cap, target_bins(cap, r, v), n_win=N_WIN)
    widths[k] = line_width(f, S)
    if k == "walker":
        spread = rl.features(t, f, S, LAM)[1]
    ax.pcolormesh(t, f, np.clip(S, -40, 0), cmap="Blues", shading="auto")
    ax.set_ylim(-600, 600); ax.set_xlabel("time (s)"); ax.set_ylabel("Doppler (Hz)")
    ax.set_title(f"{k}, record 1, {N_WIN} chirps", fontsize=10)
plt.tight_layout(); plt.show()
Tw = N_WIN * caps[("walker", 1)]["PRI"]
print(f"N_WIN = {N_WIN}: T_w = {Tw * 1e3:.0f} ms, resolution {1 / Tw:.1f} Hz")
print(f"  spinner's strongest line {widths['spinner']:.0f} Hz wide; walker's torso line {widths['walker']:.0f} Hz; "
      f"walker's spread {spread:.2f} m/s")

## Task (b): one more feature

The next cell defines `my_features`, which the classifier calls for each record. It returns the provided `features`, the centroid, spread, and periodicity in that order, and it already finds each spectrogram column's band, the bins within 15 dB of the column's peak, and the band's edge in hertz. The cell after it computes the features of all 16 records over their L30 detections' range bins, classifies each record with a nearest-centroid classifier trained on the other 15 (so no record is scored by a classifier that has seen it), and prints the confusion matrix and the accuracy.

1. Run both cells as they stand: three features. Record the matrix and the accuracy in Table E.
2. **Add one line** at the `TASK (b)` mark: append the band's **edge**, averaged over the columns and converted to m/s, the feature the lesson's Confusion Matrix frame suggests: `x = np.append(x, edge.mean() * lam / 2)`. Run both cells and record the second matrix.
3. **Change that line** to append the band's **shape** instead: the spread divided by the edge, `x[1]` over the same edge in m/s. Run both cells and record the third matrix.

**Check:** three features call 12 of 16 right, 75%: three fans (`fan_2`, `fan_3`, `fan_4`) are called spinners and one spinner (`spinner_2`) a fan, and every walker and hand is right. The edge brings it to 14 of 16, 88%, and the shape to 15 of 16, 94%. The record still wrong is `fan_3`, a slow fan held 45° off edge-on. The printed features show why both added features help, and why one record resists: compare its shape with the other fans' and the spinners', and then its spread and edge. Part 3 asks you to explain it.

In [ ]:
def my_features(t, f, S, lam):
    '''The features the classifier uses: the provided three, and any you add.'''
    x = rl.features(t, f, S, lam)                    # centroid m/s, spread m/s, periodicity
    band = S - S.max(axis=0) > -15                   # each column's band, as features() keeps it
    edge = (np.abs(f)[:, None] * band).max(axis=0)   # each column's band edge, Hz
    # TASK (b): add one feature here, e.g. x = np.append(x, ...)
    return x

In [ ]:
X, y, names = [], [], []
for (k, n), cap in caps.items():
    r, v, *_ = detect(cap)                           # the record's L30 detection
    t, f, S = rl.spectrogram(cap, target_bins(cap, r, v), n_win=64)
    X.append(my_features(t, f, S, LAM)); y.append(k); names.append(f"{k}_{n}")
X, y = np.array(X), np.array(y)
pred = leave_one_out(X, y)
M = rl.confusion_matrix(y, pred, KINDS)

print(f"{X.shape[1]} features, each record scored by a classifier trained on the other {len(y) - 1}:")
for nm, x, p in zip(names, X, pred):
    print(f"  {nm:10s} {np.array2string(x, precision=2, floatmode='fixed', suppress_small=True):34s} called {p}"
          + ("" if p == nm.split('_')[0] else "   <- wrong"))
print("\ntrue \\ called  " + "  ".join(f"{c:>7s}" for c in KINDS))
for c, row in zip(KINDS, M):
    print(f"{c:14s} " + "  ".join(f"{m:7d}" for m in row))
accuracy = np.trace(M) / M.sum()
print(f"accuracy {np.trace(M)} of {M.sum()} = {accuracy:.0%}")

plt.figure(figsize=(5, 3.5))
for k, mk in zip(KINDS, "o s ^ D".split()):
    m = y == k
    plt.scatter(X[m, 0], X[m, 1], marker=mk, facecolor="none", label=k)
plt.xlabel("centroid (m/s)"); plt.ylabel("spread (m/s)"); plt.legend(); plt.show()

## Task (c): tune the track

The cell tracks the walk of `L31_dismount.npz`. Each 128 ms frame goes through the L30 chain, the 2-pulse canceller, the map, and CFAR; the detection nearest the last one is the frame's measured range, and `monopulse_angle` in that range bin, averaged over the frame's chirps, is its angle. `alpha_beta` then smooths each with the gains $\alpha$ and $\beta = \alpha^2/(2-\alpha)$. The cell compares the track with the walk's truth, which the recorded file carries, over every frame after the first second, while the track settles. It prints the RMS error, the largest error, and the largest error during the turn at B, the lag.

1. Run it as it stands, `ALPHA = 0.5`. Record the row in Table F.
2. **Change one line:** set `ALPHA = 0.2` and run it. Record the row.
3. **Change one line:** set `ALPHA = 0.8` and run it. Record the row, then set `ALPHA` to the value you would use and run it once more; that run writes `L31_track.csv`.

**Check:** the detections alone are 0.09 m RMS from the truth in range, about a third of a range bin, and 0.29° in angle. At $\alpha = 0.5$ the range track is 0.06 m RMS, better than the detections, and lags the turn by 0.16 m; at 0.2 it is smoother frame to frame but trails the turn by 0.60 m and is still catching up after it, which makes its RMS 0.37 m; at 0.8 it follows the bins' steps, 0.07 m RMS, and lags the turn by 0.14 m. In angle the monopulse measurements are already good to a third of a degree, so smoothing buys little: the angle track's RMS falls from 2.50° at 0.2 to 0.26° and 0.19° as alpha grows, only the last below the detections'. Part 3 asks why one alpha does not suit both. Table F comes from this file, because only a record with the truth in it can score a track. A record with no truth still runs: the cell plots the track and prints its range and angle at the turn.

In [ ]:
ALPHA = 0.5         # TASK (c): change to 0.2, then 0.8, then your choice
BETA = ALPHA ** 2 / (2 - ALPHA)

# The measurements take a few seconds, so they are kept between runs, but only
# for the walk they came from: change WALK, rerun the first cell, and they are redone.
if globals().get("meas_of") != (WALK, id(walk)):
    zr, za = track_meas(walk)
    meas_of = (WALK, id(walk))
tk = (np.arange(len(zr)) + 0.5) * FRAME
trk_r = rl.alpha_beta(zr, ALPHA, BETA, FRAME)
trk_a = rl.alpha_beta(za, ALPHA, BETA, FRAME) if not np.all(np.isnan(za)) else np.full((len(za), 2), np.nan)
print(f"ALPHA = {ALPHA}, BETA = {BETA:.3f}: {len(zr)} frames, {int(np.isnan(zr).sum())} with no detection")

if "truth_t" in walk:
    Rt, At = walk["truth_r"], walk["truth_az"]
    settled = tk > 1.0
    turn = settled & (np.abs(tk - tk[np.argmax(Rt)]) < 0.8)
    er, ea = trk_r[:, 0] - Rt, trk_a[:, 0] - At
    rms_range, largest_range, lag_turn = (np.sqrt(np.mean(er[settled] ** 2)), np.abs(er[settled]).max(),
                                          np.abs(er[turn]).max())
    rms_angle, largest_angle = np.sqrt(np.mean(ea[settled] ** 2)), np.abs(ea[settled]).max()
    print(f"  detections:  range {np.sqrt(np.nanmean((zr - Rt) ** 2)):.2f} m RMS,  angle {np.sqrt(np.nanmean((za - At) ** 2)):.2f} deg RMS")
    print(f"  range track: {rms_range:.2f} m RMS, largest {largest_range:.2f} m, at the turn {lag_turn:.2f} m")
    print(f"  angle track: {rms_angle:.2f} deg RMS, largest {largest_angle:.2f} deg")
elif np.all(np.isnan(trk_r[:, 0])):
    print("  no detection in any frame: check that the record holds a walker moving toward or away from the kit")
else:
    k = int(np.nanargmax(trk_r[:, 0]))
    ang = "no iq_ch, so no angle" if np.isnan(trk_a[k, 0]) else f"{trk_a[k, 0]:+.1f} deg"
    print(f"  no truth in this record: the track turns at {trk_r[k, 0]:.2f} m, {ang}, "
          f"{tk[k]:.1f} s; compare with your tape mark B")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, z, trk, key, unit in ((axs[0], zr, trk_r, "truth_r", "range (m)"), (axs[1], za, trk_a, "truth_az", "angle (deg), + to E8")):
    if key in walk:
        ax.plot(tk, walk[key], "--", color="gray", label="truth")
    ax.plot(tk, z, "o", mfc="none", ms=4, color="k", label="detections")
    ax.plot(tk, trk[:, 0], lw=1.8, label=f"alpha = {ALPHA}")
    ax.set_xlabel("time (s)"); ax.set_ylabel(unit); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

np.savetxt("L31_track.csv", np.column_stack([tk, zr, za, trk_r, trk_a]), delimiter=",", fmt="%.3f",
           header="t_s,range_m,angle_deg,track_range_m,track_vel_mps,track_angle_deg,track_rate_dps", comments="")

## Before you close

Save the notebook with your changes and your plots, and upload the `.ipynb` with the packet. This is the last lab: keep this notebook, `L31_track.csv`, `radar_lab.py`, and your captures with the L28 to L30 work, because the report draws on all four. Anything you do beyond these three tasks belongs in your report's Extra Effort: a fifth target, a feature of your own, a classifier that weights its features, a second walker, or a tracker whose prediction steers the beam.
